In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
sns.set()
import matplotlib.pyplot as plt

from sklearn import linear_model
import lightgbm as lgbm

from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold

In [2]:
X_train = pd.read_csv('../data/X_train.csv',index_col='ROW_ID')
X_test = pd.read_csv('../data/X_test.csv',index_col='ROW_ID')

y_train = pd.read_csv('../data/y_train.csv',index_col='ROW_ID')
sample_submission = pd.read_csv('../data/sample_submission.csv',index_col='ROW_ID')
total_data = pd.concat([X_train,y_train],axis=1)

In [3]:
alloc_counts = total_data.groupby('ALLOCATION').size()
print(alloc_counts.describe())

# does an allocation's historical average target sign predict this row's target?
# (be careful: this must only use *other* rows' targets, not leak the current row)
alloc_avg_target_sign = total_data.groupby('ALLOCATION')['target'].transform(
    lambda s: np.sign(s).expanding().mean().shift(1)  # shift(1) avoids using current row
)

count     278.000000
mean     1895.946043
std       388.833385
min        19.000000
25%      1431.000000
50%      1864.000000
75%      1931.000000
max      2439.000000
dtype: float64


In [4]:
print((alloc_counts < 100).sum(), "allocations with fewer than 100 observations")

2 allocations with fewer than 100 observations


In [5]:
alloc_positive_rate = total_data.groupby('ALLOCATION')['target'].apply(
    lambda s: (np.sign(s) == 1).mean() * 100
)
print(alloc_positive_rate.describe())

count    278.000000
mean      50.593265
std        2.738922
min       38.465665
25%       49.249094
50%       50.491973
75%       51.636535
max       65.887659
Name: target, dtype: float64


In [6]:
total_data['ALLOC_HIST_POSITIVE_RATE'] = (
    total_data.groupby('ALLOCATION')['target']
    .transform(lambda s: (np.sign(s) > 0).expanding().mean().shift(1))
)

In [7]:
train_allocs = set(X_train['ALLOCATION'].unique())
test_allocs = set(X_test['ALLOCATION'].unique())
print(f"Allocations in test but not in train: {len(test_allocs - train_allocs)}")
print(f"Allocations in train but not in test: {len(train_allocs - test_allocs)}")

Allocations in test but not in train: 0
Allocations in train but not in test: 0


In [8]:
# sort by TS within each allocation so expanding() respects proper order first —
# IMPORTANT: only valid if TS order reflects true chronology per allocation
# (worth double-checking this assumption before trusting the result)
total_data_sorted = total_data.sort_values(['ALLOCATION', 'TS'])

total_data_sorted['ALLOC_HIST_POSITIVE_RATE'] = (
    total_data_sorted.groupby('ALLOCATION')['target']
    .transform(lambda s: (np.sign(s) > 0).expanding().mean().shift(1))
)

# check how it looks — early rows per allocation should have NaN or high variance,
# later rows should stabilize
print(total_data_sorted[['ALLOCATION', 'TS', 'target', 'ALLOC_HIST_POSITIVE_RATE']].head(20))

print(f"\nNaN count (first observation per allocation): {total_data_sorted['ALLOC_HIST_POSITIVE_RATE'].isna().sum()}")

           ALLOCATION         TS    target  ALLOC_HIST_POSITIVE_RATE
ROW_ID                                                              
0       ALLOCATION_01  DATE_0001  0.009210                       NaN
276     ALLOCATION_01  DATE_0002  0.007241                  1.000000
825     ALLOCATION_01  DATE_0005  0.008570                  1.000000
1166    ALLOCATION_01  DATE_0007  0.009459                  1.000000
1717    ALLOCATION_01  DATE_0010 -0.003446                  1.000000
2474    ALLOCATION_01  DATE_0014  0.003327                  0.800000
2750    ALLOCATION_01  DATE_0015  0.007531                  0.833333
3026    ALLOCATION_01  DATE_0016 -0.000196                  0.857143
3302    ALLOCATION_01  DATE_0017  0.003256                  0.750000
3782    ALLOCATION_01  DATE_0019 -0.001220                  0.777778
4256    ALLOCATION_01  DATE_0023 -0.001969                  0.700000
4670    ALLOCATION_01  DATE_0026 -0.007629                  0.636364
4946    ALLOCATION_01  DATE_0027  

In [9]:
# check if RET_1 at row t+1 matches RET_2 at row t for the same allocation
# (if TS order is truly chronological, yesterday's "2-days-ago" return
#  should equal today's "1-day-ago" return, shifted by one row)
alloc01 = total_data_sorted[total_data_sorted['ALLOCATION'] == 'ALLOCATION_01']
comparison = pd.DataFrame({
    'RET_2_at_t': alloc01['RET_2'],
    'RET_1_at_t_plus_1': alloc01['RET_1'].shift(-1)
})
print(comparison.head(15))
print(f"\nCorrelation: {comparison['RET_2_at_t'].corr(comparison['RET_1_at_t_plus_1']):.4f}")

        RET_2_at_t  RET_1_at_t_plus_1
ROW_ID                               
0        -0.000178           0.005110
276      -0.004439           0.005628
825      -0.000890          -0.003151
1166     -0.004065           0.005724
1717     -0.006574          -0.002946
2474     -0.003006          -0.001909
2750      0.002221           0.002597
3026     -0.002736          -0.000375
3302      0.000881          -0.001959
3782      0.001039           0.001064
4256      0.005839           0.000855
4670     -0.002083           0.002016
4946      0.000879           0.007481
5222      0.001861           0.002346
5428     -0.000945           0.002789

Correlation: 0.0073


In [10]:
alloc01 = total_data[total_data['ALLOCATION'] == 'ALLOCATION_01'].copy()

ret_cols = [f'RET_{i}' for i in range(1, 21)]

# For each pair of rows (A, B) for this allocation, check if A's older lags
# match B's newer lags shifted by some offset — i.e. does RET_5 at row A
# equal RET_1 at some row 4 "steps" later in true time?
# Simplify: just check adjacent-by-index overlap at a few offsets to see if ANY
# shift value gives high correlation (not just offset=1 which we already ruled out)

for offset in range(1, 6):
    ret_earlier = alloc01[f'RET_{offset+1}'].reset_index(drop=True)  # "offset+1 days ago" at row t
    ret_later = alloc01['RET_1'].shift(-offset).reset_index(drop=True)  # "1 day ago" at row t+offset
    corr = ret_earlier.corr(ret_later)
    print(f"offset={offset}: corr(RET_{offset+1} at t, RET_1 at t+{offset}) = {corr:.4f}")

offset=1: corr(RET_2 at t, RET_1 at t+1) = 0.0073
offset=2: corr(RET_3 at t, RET_1 at t+2) = 0.0075
offset=3: corr(RET_4 at t, RET_1 at t+3) = 0.0035
offset=4: corr(RET_5 at t, RET_1 at t+4) = 0.0144
offset=5: corr(RET_6 at t, RET_1 at t+5) = -0.0276


In [11]:
alloc_vol_full = total_data.groupby('ALLOCATION')['target'].apply(lambda s: s.std())
print(alloc_vol_full.describe())

count    278.000000
mean       0.003016
std        0.001479
min        0.001070
25%        0.002247
50%        0.002790
75%        0.003489
max        0.016293
Name: target, dtype: float64


In [12]:
minimal_features_vol = ['RET_1', 'MEDIAN_DAILY_TURNOVER', 'ALLOC_TARGET_ENCODING', 'ALLOC_VOLATILITY']
# (reuse your existing CV loop, adding ALLOC_TARGET_ENCODING + ALLOC_VOLATILITY together
#  inside the fold loop the same way you already do for ALLOC_TARGET_ENCODING)